In [ ]:
# ===============================
# 1. Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer


# ===============================
# 2. Load your dataset
# ===============================
# Replace with your actual dataset path
heart_df = pd.read_csv("./data/heart_2020_cleaned.csv")


df = heart_df.copy()

# --- Risk Score Calculation ---

df["RiskScore"] = 0

# 1. Very strong medical risks (weight = 4)
df["RiskScore"] += (df["Stroke"] == "Yes") * 4
df["RiskScore"] += (df["Diabetic"] == "Yes") * 4
df["RiskScore"] += (df["KidneyDisease"] == "Yes") * 4

# 2. Strong lifestyle/functional risks (weight = 3)
df["RiskScore"] += (df["Smoking"] == "Yes") * 3
df["RiskScore"] += (df["AlcoholDrinking"] == "Yes") * 3
df["RiskScore"] += (df["DiffWalking"] == "Yes") * 3

# 3. Moderate risks (weight = 2)
df["RiskScore"] += (df["Asthma"] == "Yes") * 2
df["RiskScore"] += (df["PhysicalActivity"] == "No") * 2

# BMI: assuming BMI is categorical numeric (1-6)
df["RiskScore"] += (df["BMI"] >= 3) * 2   # 3: overweight, 4-6: obese levels

# 4. Mild risks (weight = 1)
df["RiskScore"] += (df["SleepTime"] < 6) * 1
df["RiskScore"] += (df["SleepTime"] > 9) * 1
df["RiskScore"] += (df["PhysicalHealth"] > 10) * 1
df["RiskScore"] += (df["MentalHealth"] > 10) * 1

# 5. GenHealth (ordinal weights)
gen_map = {
    "Poor": 4,
    "Fair": 3,
    "Good": 2,
    "Very good": 1,
    "Excellent": 0
}
df["RiskScore"] += df["GenHealth"].map(gen_map)

# 6. AgeCategory (ordinal weights)
age_map = {
    "18-24": 0,
    "25-29": 1,
    "30-34": 1,
    "35-39": 1,
    "40-44": 2,
    "45-49": 2,
    "50-54": 3,
    "55-59": 3,
    "60-64": 4,
    "65-69": 4,
    "70-74": 5,
    "75-79": 5,
    "80 or older": 6
}
# RiskScore column
df["RiskScore"] += df["AgeCategory"].map(age_map)

def categorize_risk(score):
    if score <= 3:
        return "Very Low"
    elif score <= 6:
        return "Low"
    elif score <= 10:
        return "Moderate"
    elif score <= 15:
        return "High"
    else:
        return "Very High"
# RiskCategory column
df["RiskCategory"] = df["RiskScore"].apply(categorize_risk)


X = df.drop(["RiskCategory", "RiskScore"], axis=1)
y = df["RiskScore"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===============================
# 4. Preprocessing
# ===============================
# -----------------------------
# ORDINAL COLUMNS & CATEGORIES
# -----------------------------

ordinal_cols = ["AgeCategory", "GenHealth", "Diabetic"]

ordinal_categories = [

    # AgeCategory order
    ["18-24", "25-29", "30-34", "35-39", "40-44", "45-49",
     "50-54", "55-59", "60-64", "65-69", "70-74", "75-79", "80 or older"],

    # GenHealth order
    ["Poor", "Fair", "Good", "Very good", "Excellent"],

    # Diabetic order
    ["No", "No, borderline diabetes", "Yes", "Yes (during pregnancy)"]
]


# -----------------------------
# Nominal categorical columns
# -----------------------------

nominal_cols = [
    'Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 'Sex',
    'Race', 'PhysicalActivity', 'Asthma',
    'KidneyDisease', 'SkinCancer'
]

# -----------------------------
# Numeric columns
# -----------------------------

num_cols = ['BMI', 'PhysicalHealth', 'MentalHealth', 'SleepTime']

# -----------------------------
# PIPELINES
# -----------------------------

# Numeric pipeline
num_pipe = Pipeline([
    ("scaler", StandardScaler())
])

# Ordinal pipeline
ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=ordinal_categories))
])

# Nominal pipeline
nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# -----------------------------
# COMBINED PREPROCESSOR
# -----------------------------

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("ord", ordinal_pipe, ordinal_cols),
    ("nom", nominal_pipe, nominal_cols)
])

# ===============================
# 5. Gradient Booster Regressor
# ===============================

pipe = Pipeline([
    ("prep", preprocessor),               # your existing preprocessor
    ("model", GradientBoostingRegressor(learning_rate=0.2, max_depth=4, n_estimators=300)) # used best params from GridSearchCV
])

pipe.fit(X_train, y_train)

preds = pipe.predict(X_test)

mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, preds)

print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# ===============================
# 8. Convert to Risk Percentage & Category
# ===============================
# Convert probability to percentage

max_score = 40                            # maximum weight in RiskScore
risk_percentage = (preds / max_score) * 100

# Define risk categories
def convert_percentage_to_category(p):

    if p < 10:
        return "Very Low"
    elif p < 25:
        return "Low"
    elif p < 40:
        return "Moderate"
    elif p < 60:
        return "High"
    else:
        return "Very High"

preds_cat = [convert_percentage_to_category(p) for p in risk_percentage]

# Combine into a DataFrame for display
risk_results = pd.DataFrame({
    "Risk Scores": preds,
    "Risk Percentage": risk_percentage,
    "Risk Category": preds_cat
})

# Show first few results
risk_results.head()


MAE:  0.0661
MSE:  0.0112
RMSE: 0.1059
R²:   0.9996


,Predicted Probability,Risk Percentage,Risk Category
0,8.047447,20.118616,Low
1,4.010533,10.026333,Low
2,9.974757,24.936894,Low
3,11.023383,27.558458,Moderate
4,21.918377,54.795943,High
